In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from pathlib import Path

# ============ CONFIG ============
BASE_MODEL_HF = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # Original base model
OUTPUT_DIR_BASE_8BIT = "/home/spark/projects/stoic/output/vllm/llama31_8b_instruct_base_8bit"
# ================================

print(f"💾 Creating 8-bit quantized BASE Llama 3.1 8B model...")
print(f"   Source: {BASE_MODEL_HF}")
print(f"   Output: {OUTPUT_DIR_BASE_8BIT}")

# Configure 8-bit quantization
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

print(f"📥 Loading base model with 8-bit quantization...")
model_8bit_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_HF,
    quantization_config=bnb_config_8bit,
    device_map="auto",
    torch_dtype=torch.float16,
)
tokenizer_8bit_base = AutoTokenizer.from_pretrained(BASE_MODEL_HF)

print(f"💾 Saving 8-bit base model...")
Path(OUTPUT_DIR_BASE_8BIT).mkdir(parents=True, exist_ok=True)
model_8bit_base.save_pretrained(OUTPUT_DIR_BASE_8BIT)
tokenizer_8bit_base.save_pretrained(OUTPUT_DIR_BASE_8BIT)

print(f"✅ 8-bit base model saved!")
print(f"\n📊 Use with vLLM + LoRA adapters:")
print(f"   vllm serve {OUTPUT_DIR_BASE_8BIT} \\")
print(f"     --quantization bitsandbytes \\")
print(f"     --enable-lora \\")
print(f"     --lora-modules biblical=/home/spark/projects/stoic/output/biblical_llama31_8b_instruct_unsloth/train/checkpoint-1151 \\")
print(f"     --max-lora-rank 8")
print(f"\n📊 Total VRAM: ~9 GB (base) + 0.2 GB (adapters) = ~9.2 GB")

## Save 8-bit Base Model for vLLM + LoRA Adapters

This notebook section saves an **8-bit quantized base Llama 3.1 8B model** that can be used with LoRA adapters in vLLM.

**Setup:**
- 8-bit base model: ~9 GB VRAM
- LoRA adapters: 218 MB (fp16, already saved)
- Total: ~9.2 GB VRAM with adapters loaded

# Biblical Llama 3.1 Fine-Tuning with Unsloth

**Base Model:** Llama 3.1 8B Instruct

**Dataset:** Augmentoolkit-generated Biblical persona (first-person apostolic responses)

**Chat Template:** `<|begin_of_text|><|start_header_id|>role<|end_header_id|>content<|eot_id|>`

## Step 1: Configuration

All paths and variables for easy configuration.

In [ ]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================

# =========================== MODEL CONFIGURATION ===========================
# Base model to fine-tune
BASE_LLM = "unsloth/Meta-Llama-3.1-8B-Instruct"

# Name for output folders and model identification
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth"

# =========================== INPUT DATA ===========================
# Path to training data (Augmentoolkit output)
INPUT_DATA_PATH = "/home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset"

# =========================== OUTPUT DIRECTORIES ===========================
# All outputs organized under ./output/{MODEL_NAME_BASE}/
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"       # LoRA weights during training
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model"         # Full merged model
OUTPUT_DIR_GGUF = f"{OUTPUT_BASE_DIR}/gguf"            # GGUF for Ollama

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 2048    # Max tokens per example
BATCH_SIZE = 4           # Per-device batch size
GRAD_ACCUM = 4           # Gradient accumulation
LEARNING_RATE = 5e-5     # Learning rate (lower = gentler training)
TARGET_EPOCHS = 1        # Number of training epochs

# =========================== LoRA CONFIGURATION ===========================
# LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training
# small adapter matrices instead of full model weights.
#
# LORA_R (Rank): Controls adapter capacity
#   - Lower (4-8): Less aggressive, preserves base model behavior
#   - Higher (16-64): More capacity for new knowledge, risk of overfitting
#   - Recommendation: Start with 8, increase if underfitting
#
# LORA_ALPHA: Scaling factor for LoRA weights
#   - Typically set to 2x LORA_R (e.g., r=8 → alpha=16)
#   - Higher alpha = stronger influence of fine-tuning
#
# LORA_DROPOUT: Regularization during training
#   - 0: No dropout (faster training)
#   - 0.05-0.1: Mild regularization for larger datasets
#
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# =========================== LoRA TARGET MODULES ===========================
# Which layers to fine-tune. Llama architecture has these trainable layers:
#
# ATTENTION MODULES (Recommended for persona/style transfer)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ q_proj       │ Query projection        │ "What am I looking for?"            │
# │              │                         │ Core attention steering             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ k_proj       │ Key projection          │ "What info do I have?"              │
# │              │                         │ Memory/context matching             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ v_proj       │ Value projection        │ "What info to pass forward?"        │
# │              │                         │ Content selection                   │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ o_proj       │ Output projection       │ Combines attention heads            │
# │              │                         │ Post-attention mixing               │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# MLP/FFN MODULES (More aggressive - use for knowledge injection)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ gate_proj    │ Gate projection         │ Controls information flow           │
# │              │                         │ Heavy knowledge modification        │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ up_proj      │ Up projection           │ Expands to hidden dimension         │
# │              │                         │ Feature extraction                  │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ down_proj    │ Down projection         │ Compresses back to model dim        │
# │              │                         │ Output generation                   │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# PRESETS:
#   Conservative (style/persona):  ["q_proj", "v_proj"]
#   Balanced:                      ["q_proj", "k_proj", "v_proj", "o_proj"]
#   Aggressive (new knowledge):    ["q_proj", "k_proj", "v_proj", "o_proj", 
#                                   "gate_proj", "up_proj", "down_proj"]
#
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # Conservative: attention-only

# =========================== GGUF CONVERSION ===========================
# Path to llama.cpp installation (for GGUF conversion)
LLAMA_CPP_PATH = "/home/spark/resources/llama.cpp"

# Quantization type for GGUF export
# Options: None (fp32), "q4", "q8", "q4_k", "q5_k", "q6_k", "fp16"
# Recommendation: "q8" for best quality/size balance
QUANTIZATION_TYPE = "q8"

# =========================== VLLM DEPLOYMENT ===========================
# Centralized vLLM models directory (mapped to /models in container)
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"

# =========================== INFERENCE TEST ===========================
# Test prompt for inference validation
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("✓ Configuration loaded")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")

## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [ ]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Biblical dataset (first-person apostolic responses from Paul, David, Peter, etc.).

In [ ]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ALL subdirectories and ALL files
all_dirs = glob.glob(f"{INPUT_DATA_PATH}/*/")

print("📚 LOADING ALL AUGMENTOOLKIT DATA")
print(f"Found {len(all_dirs)} subdirectories")

datasets = []
for dir_path in sorted(all_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        try:
            ds = load_dataset("json", data_files=file_path, split="train")
            datasets.append(ds)
            print(f"  Loaded {len(ds)} from {dir_path.split('/')[-2]}/{file_path.split('/')[-1]}")
        except Exception as e:
            print(f"  Skipped {file_path.split('/')[-1]}: {e}")

dataset = concatenate_datasets(datasets)

print(f"\n✓ Total dataset loaded: {len(dataset)} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n--- First Example (first 800 chars) ---")
import json
print(json.dumps(dataset[0], indent=2)[:800])

# Shuffle dataset for better training
dataset = dataset.shuffle(seed=42)
print(f"\n✓ Dataset shuffled and ready for training")

## 4. Load Model & Tokenizer with Unsloth

Load Llama 3.1 8B Instruct model in FULL 16-bit precision (no quantization).

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load model in FULL 16-bit precision (no quantization)
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,  # Full 16-bit precision
    load_in_4bit=False,   # NO quantization
    device_map={"": 0}    # Force all on GPU 0
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"✓ Precision: FULL 16-bit (fp16)")
print(f"✓ Tokenizer configured")
print(f"✓ Max sequence length: {MAX_SEQ_LENGTH}")

In [ ]:
# Format dataset for Llama 3.1 chat template
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n✓ Dataset formatted: {len(dataset)} examples")


## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See configuration section for module explanations.

In [ ]:
from peft import LoraConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"✓ LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Biblical examples from epistles and psalms
- 1 epoch with low learning rate (5e-5) to gently teach persona
- This preserves base model capabilities while adding authentic voice

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = min(200, steps_per_epoch)  # Save every 200 steps or at epoch end

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=True,
        bf16=False,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for PURE DOMAIN (light training)")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {TARGET_EPOCHS}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps (every epoch)")

In [ ]:
# Start training
trainer.train()

## 7. Save Model

In [ ]:
# Save LoRA adapters first (small, reusable on base model)
print(f"💾 Saving LoRA adapters to {OUTPUT_DIR_ADAPTERS}...")
model.save_pretrained(OUTPUT_DIR_ADAPTERS)
tokenizer.save_pretrained(OUTPUT_DIR_ADAPTERS)
print(f"✓ LoRA adapters saved to {OUTPUT_DIR_ADAPTERS}")

# Save merged model (standalone, ready for deployment)
print(f"\n💾 Saving merged model to {OUTPUT_DIR_MERGED}...")
model.save_pretrained_merged(OUTPUT_DIR_MERGED, tokenizer, save_method="merged_16bit")
print(f"✓ Merged model saved to {OUTPUT_DIR_MERGED}")

## 7b. Deploy to vLLM

Copy the merged model to the centralized vLLM models folder for OpenWebUI access.

In [ ]:
import shutil
from pathlib import Path

# ============ STANDALONE CONFIG ============
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth"
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model"
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"
# ============================================

VLLM_MODEL_PATH = Path(VLLM_MODELS_DIR) / MODEL_NAME_BASE

# Create vllm directory if needed
Path(VLLM_MODELS_DIR).mkdir(parents=True, exist_ok=True)

# Copy merged model to vLLM folder
if VLLM_MODEL_PATH.exists():
    print(f"⚠️  Removing existing model at {VLLM_MODEL_PATH}")
    shutil.rmtree(VLLM_MODEL_PATH)

print(f"📦 Copying merged model to vLLM folder...")
print(f"   From: {OUTPUT_DIR_MERGED}")
print(f"   To:   {VLLM_MODEL_PATH}")


shutil.copytree(OUTPUT_DIR_MERGED, VLLM_MODEL_PATH)print(f"\n   Or via OpenWebUI, add as OpenAI-compatible endpoint at http://localhost:8000/v1")

print(f"   vllm serve /models/{MODEL_NAME_BASE}")

print(f"\n✅ Model deployed to vLLM!")print(f"\n🚀 To serve this model, update docker-compose command to:")
print(f"   Container path: /models/{MODEL_NAME_BASE}")

## 7c. Test Inference

In [ ]:
from unsloth import FastLanguageModel
import torch

# ============ STANDALONE CONFIG ============
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth"
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model"
MAX_SEQ_LENGTH = 2048
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"
# ============================================

# Load model if not already loaded
if 'model' not in dir() or model is None:
    print(f"📥 Loading model from {OUTPUT_DIR_MERGED}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=OUTPUT_DIR_MERGED,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=False,
    )

# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test inference with a Biblical question
inputs = tokenizer.apply_chat_template(

    [{"role": "user", "content": TEST_PROMPT}],print(f"Assistant: {assistant_response}")

    add_generation_prompt=True,print(f"User: {TEST_PROMPT}")

    return_tensors="pt"assistant_response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].split("<|eot_id|>")[0].strip()

).to("cuda")# Llama 3 uses <|eot_id|> as end token

print("\n=== PARSED OUTPUT ===")

outputs = model.generate(print(response)

    input_ids=inputs,print("\n=== RAW OUTPUT ===")

    max_new_tokens=256,

    temperature=0.7, response = tokenizer.decode(outputs[0], skip_special_tokens=False)

    top_p=0.9,)
    repetition_penalty=1.1

## Notes

### Dataset Quality
This notebook uses Augmentoolkit-generated data from Biblical texts (Paul's epistles, David's psalms, etc.). The pipeline enforced first-person apostolic responses through configuration:
- `shared_instruction`: "You ARE a Biblical speaker - not explaining Scripture, but EMBODYING it."
- Source texts: Paul, Peter, James, John (epistles), David (Psalms), prophets
- All prompts rewritten to enforce "I am an apostle..." or "I have witnessed..." voice

### Next Steps
- For broader coverage: Run Augmentoolkit with additional Biblical books
- Merge datasets: Combine with other first-person Biblical writings
- Evaluate first-person quality: Test if model says "I have witnessed..." vs "The Bible teaches..."

### Dataset Location
Augmentoolkit output: `/home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset/sft_run/combined_factual_data/`


## 7d. Save Quantized Safetensors Model

Save a 4-bit quantized version in safetensors format for efficient deployment with vLLM, TGI, or other frameworks that support bitsandbytes 4-bit quantization.

**Quantization Methods:**
- `"merged_4bit_forced"`: 4-bit bitsandbytes quantization - smaller size, faster inference
- `"merged_16bit"`: 16-bit float - high quality (this is what we saved in step 7)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from pathlib import Path

# Uses config from Step 1: MODEL_NAME_BASE, OUTPUT_BASE_DIR, OUTPUT_DIR_MERGED
OUTPUT_DIR_SAFETENSORS_QUANT = f"{OUTPUT_BASE_DIR}/{MODEL_NAME_BASE}_4bit"

print(f"💾 Creating 4-bit quantized safetensors model...")
print(f"   Source: {OUTPUT_DIR_MERGED}")
print(f"   Output: {OUTPUT_DIR_SAFETENSORS_QUANT}")

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"📥 Loading model with 4-bit quantization...")
model_4bit = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR_MERGED,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tokenizer_4bit = AutoTokenizer.from_pretrained(OUTPUT_DIR_MERGED)

print(f"💾 Saving quantized model...")
Path(OUTPUT_DIR_SAFETENSORS_QUANT).mkdir(parents=True, exist_ok=True)
model_4bit.save_pretrained(OUTPUT_DIR_SAFETENSORS_QUANT)
tokenizer_4bit.save_pretrained(OUTPUT_DIR_SAFETENSORS_QUANT)

print(f"✅ 4-bit quantized safetensors model saved!")
print(f"\n📊 This model can be loaded with:")
print(f"   from transformers import AutoModelForCausalLM")
print(f"   model = AutoModelForCausalLM.from_pretrained('{OUTPUT_DIR_SAFETENSORS_QUANT}', device_map='auto')")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from pathlib import Path

# Uses config from Step 1: MODEL_NAME_BASE, OUTPUT_BASE_DIR, OUTPUT_DIR_MERGED
OUTPUT_DIR_8BIT = f"{OUTPUT_BASE_DIR}/{MODEL_NAME_BASE}_8bit"

print(f"💾 Creating 8-bit quantized model...")
print(f"   Source: {OUTPUT_DIR_MERGED}")
print(f"   Output: {OUTPUT_DIR_8BIT}")

# Configure 8-bit quantization
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

print(f"📥 Loading model with 8-bit quantization...")
model_8bit = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR_MERGED,
    quantization_config=bnb_config_8bit,
    device_map="auto",
    torch_dtype=torch.float16,
)
tokenizer_8bit = AutoTokenizer.from_pretrained(OUTPUT_DIR_MERGED)

print(f"💾 Saving 8-bit quantized model...")
Path(OUTPUT_DIR_8BIT).mkdir(parents=True, exist_ok=True)
model_8bit.save_pretrained(OUTPUT_DIR_8BIT)
tokenizer_8bit.save_pretrained(OUTPUT_DIR_8BIT)

print(f"✅ 8-bit quantized model saved!")
print(f"\n📊 VRAM comparison:")
print(f"   4-bit: ~5 GB  | 8-bit: ~9 GB  | 16-bit: ~18 GB")
print(f"\n📊 To use with vLLM:")
print(f"   vllm serve {OUTPUT_DIR_8BIT} --quantization bitsandbytes")

## 7e. Save 8-bit Quantized Model (Alternative)

Save an 8-bit quantized version as an alternative to 4-bit. 8-bit provides better quality with less rambling, while still reducing VRAM usage significantly compared to 16-bit.

**Comparison:**
- **4-bit**: ~5 GB VRAM, fastest inference, may have quality issues (rambling)
- **8-bit**: ~9 GB VRAM, better quality, reduced rambling ⭐ **Good balance**
- **16-bit**: ~18 GB VRAM, best quality, no quantization artifacts

## 8. Convert to GGUF for Ollama

**Quantization Options:**
- `None`: Full precision (fp32) - maximum quality, largest size
- `"q4"`: 4-bit (Q4_0) - highest compression, fastest inference
- `"q8"`: 8-bit (Q8_0) - best balance ⭐ **Recommended**
- `"q4_k"`: 4-bit K-quant (Q4_K_M) - better quality than q4
- `"q5_k"`: 5-bit K-quant (Q5_K_M) - excellent quality
- `"q6_k"`: 6-bit K-quant (Q6_K) - near-lossless

In [ ]:
from pathlib import Path
import subprocess
import sys

# ============ STANDALONE CONFIG ============
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth"
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model"
OUTPUT_DIR_GGUF = f"{OUTPUT_BASE_DIR}/gguf"
LLAMA_CPP_PATH = "/home/spark/resources/llama.cpp"
QUANTIZATION_TYPE = "q8"  # Options: None, "q4", "q8", "q4_k", "q5_k", "q6_k", "fp16"
# ============================================

quant_suffix = "fp32" if QUANTIZATION_TYPE is None else QUANTIZATION_TYPE
OUTPUT_DIR_GGUF_FULL = f"{OUTPUT_DIR_GGUF}/{quant_suffix}"
Path(OUTPUT_DIR_GGUF_FULL).mkdir(parents=True, exist_ok=True)

print(f"🔧 GGUF Conversion for Ollama")
print(f"   Source model: {OUTPUT_DIR_MERGED}")
print(f"   Quantization: {QUANTIZATION_TYPE or 'Full Precision (fp32)'}")
print(f"   Output: {OUTPUT_DIR_GGUF_FULL}")

# Verify llama.cpp exists

if not Path(LLAMA_CPP_PATH).exists():print(f"✓ llama.cpp found at {LLAMA_CPP_PATH}")

    raise FileNotFoundError(

        f"❌ llama.cpp not found at {LLAMA_CPP_PATH}\n"    )

        f"   This DGX uses a shared resources folder.\n"        f"   Then build it: cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build -j$(nproc)"
        f"   Please clone it: git clone https://github.com/ggerganov/llama.cpp {LLAMA_CPP_PATH}\n"

In [ ]:
from pathlib import Path
import subprocess
import sys

# ============ STANDALONE CONFIG ============
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth"
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model"
OUTPUT_DIR_GGUF = f"{OUTPUT_BASE_DIR}/gguf"
LLAMA_CPP_PATH = "/home/spark/resources/llama.cpp"
QUANTIZATION_TYPE = "q8"
quant_suffix = "fp32" if QUANTIZATION_TYPE is None else QUANTIZATION_TYPE
OUTPUT_DIR_GGUF_FULL = f"{OUTPUT_DIR_GGUF}/{quant_suffix}"
# ============================================

if QUANTIZATION_TYPE is None:
    print("\n[Step 1/1] Converting to full-precision GGUF (fp32)...")
    FINAL_GGUF = Path(OUTPUT_DIR_GGUF_FULL) / f"{MODEL_NAME_BASE}-fp32.gguf"
    subprocess.run([
        sys.executable,
        str(Path(LLAMA_CPP_PATH) / "convert_hf_to_gguf.py"),
        OUTPUT_DIR_MERGED, "--outfile", str(FINAL_GGUF), "--outtype", "f32",
    ], check=True)
else:
    print("\n[Step 1/2] Converting to fp16 GGUF...")
    TEMP_GGUF = Path(OUTPUT_DIR_GGUF_FULL) / "temp-fp16.gguf"
    subprocess.run([
        sys.executable,
        str(Path(LLAMA_CPP_PATH) / "convert_hf_to_gguf.py"),
        OUTPUT_DIR_MERGED, "--outfile", str(TEMP_GGUF), "--outtype", "f16",
    ], check=True)
    
    print(f"\n[Step 2/2] Quantizing to {QUANTIZATION_TYPE}...")
    FINAL_GGUF = Path(OUTPUT_DIR_GGUF_FULL) / f"{MODEL_NAME_BASE}-{QUANTIZATION_TYPE}.gguf"

    quant_map = {"q4": "Q4_0", "q8": "Q8_0", "q4_k": "Q4_K_M", "q5_k": "Q5_K_M", "q6_k": "Q6_K", "fp16": "F16"}print(f"\n✅ GGUF conversion complete: {FINAL_GGUF}")

    llama_quant_type = quant_map.get(QUANTIZATION_TYPE, QUANTIZATION_TYPE.upper())

    quantize_binary = Path(LLAMA_CPP_PATH) / "build" / "bin" / "llama-quantize"    TEMP_GGUF.unlink()

        subprocess.run([str(quantize_binary), str(TEMP_GGUF), str(FINAL_GGUF), llama_quant_type], check=True)

    if not quantize_binary.exists():    

        raise FileNotFoundError(        )

            f"❌ llama-quantize binary not found at {quantize_binary}\n"            f"   cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j$(nproc)"
            f"   Please build llama.cpp:\n"

In [ ]:
from pathlib import Path

# ============ STANDALONE CONFIG ============
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth"
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_GGUF = f"{OUTPUT_BASE_DIR}/gguf"
QUANTIZATION_TYPE = "q8"
quant_suffix = "fp32" if QUANTIZATION_TYPE is None else QUANTIZATION_TYPE
OUTPUT_DIR_GGUF_FULL = f"{OUTPUT_DIR_GGUF}/{quant_suffix}"
FINAL_GGUF = Path(OUTPUT_DIR_GGUF_FULL) / f"{MODEL_NAME_BASE}-{QUANTIZATION_TYPE}.gguf"
# ============================================

# Create Modelfile for Ollama - Llama 3 format
print("\n📝 Creating Modelfile for Ollama...")

MODELFILE_PATH = Path(OUTPUT_DIR_GGUF_FULL) / "Modelfile"

# Llama 3 chat template with proper special tokens
modelfile_content = f'''FROM ./{FINAL_GGUF.name}

TEMPLATE """<|start_header_id|>system<|end_header_id|>

{{{{ .System }}}}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{{{ .Prompt }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>


"""print(f"   ollama create {MODEL_NAME_BASE} -f Modelfile")

print(f"   cd {OUTPUT_DIR_GGUF_FULL}")

PARAMETER stop "<|eot_id|>"print(f"\n🚀 To import into Ollama:")

PARAMETER stop "<|end_of_text|>"print(f"✅ Modelfile created: {MODELFILE_PATH}")

'''

MODELFILE_PATH.write_text(modelfile_content, encoding="utf-8")